# Extension: MC Dropout uncertainty and a volatility-regime scanner for MT

MT (and the paper's models generally) output a single probability per factor per month with
no sense of how much to trust that number, and no rule for when to sit out entirely. This
notebook builds two independent ways to add that missing risk signal — one that asks the
*model* how confident it is, the other that ignores the model and asks the *return series*
whether the environment is currently calm or turbulent — and compares four resulting
strategies against MT's unweighted baseline.

## Idea 1: MC Dropout — ask the model

Monte Carlo Dropout (Gal and Ghahramani, 2016) is a way to get an uncertainty estimate
essentially for free from a network that already uses dropout for regularization.

**Normally**, dropout randomly zeroes a fraction of neurons *during training only*, so the
network can't lean on any single narrow path through its own representation — it's switched
off at prediction time so the network's full capacity is used.

**MC Dropout's reframe**: leave dropout switched *on* at prediction time, and instead of asking
the network once, ask it the same question 30-100 times. Each pass silences a different random
subset of neurons, so each pass can give a slightly different answer — not because the world
changed, but because each pass samples a slightly different internal path through what the
network learned. If the input sits in territory the model understands well, the passes agree
closely. If it doesn't, the passes scatter. That scatter is the signal: it's not "what's the
answer," it's "how much do internally-varied versions of my own reasoning agree with each
other" — an empirical, after-the-fact check on whether the model has a stable, well-supported
answer or is essentially guessing behind a confident-looking number.

## Idea 2: volatility scanner — ask the return series

A simpler, model-free companion: instead of perturbing the network, look at each factor's own
trailing realized volatility (rolling 3-month std of its return) — no model, no look-ahead,
only what's already known by the signal date. Flag, and abstain on, a factor-month whenever
that trailing volatility exceeds a cutoff calibrated per fold from that fold's validation-period
data. This tests a different hypothesis from MC Dropout: not "does the model doubt itself" but
"is the recent environment turbulent" — and, per the comparison at the end of this notebook, it
turns out to catch a mostly disjoint set of risky months (see the README for the headline
result: of the two, the volatility scanner is the one that actually beats the baseline).

**What this notebook does:**
1. Train an MT variant with dropout layers in the shared trunk — same walk-forward procedure
   as `Initial MT.ipynb`, nothing else changes.
2. At prediction time, run each month's predictors through the network 50 times with dropout
   forced on, per factor. The mean of those 50 passes is the point estimate (same role as MT's
   usual single output); the spread (std) is the MC-Dropout uncertainty signal.
3. Calibrate two cutoffs each fold, both from that fold's validation-period data only — never
   test, to avoid leakage: an MC-Dropout uncertainty cutoff (e.g. flag the most-uncertain 20% of
   factor-months), and a separate trailing-volatility cutoff for the scanner.
4. Compare four strategies against the unweighted baseline: two ways of acting on MC-Dropout
   uncertainty (shrink position size in proportion to uncertainty, or sit out flagged months
   entirely) and the volatility scanner's abstain rule — then check whether the scanner is just
   re-flagging the months MC Dropout already flags, or catching something genuinely different.

Architecture note: the paper's MT uses batch normalization, not dropout — this is a deliberate
addition for this extension. Forcing `training=True` to activate dropout at prediction time
would *also* put batch norm into batch-statistics mode (a well-known MC-Dropout pitfall), so
this variant's shared trunk uses dropout in place of batch norm instead of combining both.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

import loading
import estimation as est

FACTOR_NAMES = est.FACTOR_NAMES

In [2]:
# Same walk-forward estimation procedure as Initial MT.ipynb (paper Sec 3.2.4) — this notebook
# only changes what happens at *prediction* time, not the training/validation/test split.

data, feature_cols = est.build_labels_and_panel(
    loading.response_factors, loading.macro_predictors, loading.financial_predictors
)

print("data shape:", data.shape)
print("features:", len(feature_cols))
print("date range:", data.index.min().date(), "to", data.index.max().date())

data shape: (683, 264)
features: 259
date range: 1965-01-31 to 2021-11-30


In [3]:
N_MC_SAMPLES = 50          # forward passes per prediction; paper's own NN ensembling uses 10 seeds, 30-100 is the usual MC Dropout range
UNCERTAINTY_QUANTILE = 0.80  # flag the most-uncertain 20% of factor-months, calibrated per fold on validation data

# l1=0.01 (the value used by every other neural net in this project, and the smallest value in
# the paper's own Table IA1 grid for NN/LSTM/MT models {0.005, 0.007, 0.01, 0.02}) collapses this
# model's shared-trunk weights to ~0 during training: the plain MT model in Initial MT.ipynb uses
# the same l1 without issue because batch norm renormalizes activations regardless of raw weight
# scale, but this variant replaces batch norm with dropout (see markdown above), so nothing
# counteracts the L1 penalty and Adam just drives the input-dependent weights to zero, leaving
# predictions to be carried almost entirely by the (unregularized) bias. That makes the network
# effectively input-independent, so different dropout masks barely move the output and MC-Dropout
# uncertainty collapses to float32 noise (~1e-7) instead of a real signal -- confirmed empirically
# (shared_dense_1 max|weight| ~0.0005 at l1=0.01, vs ~0.16-0.28 once l1 is small enough). Checked
# every value in the paper's grid directly: 0.005 also collapses in most folds tested. l1=0.001
# (off-grid, but the smallest deviation that reliably avoided collapse across every fold tested)
# is used for this model only -- a deliberate, documented departure from the paper's grid, driven
# by the batch-norm-to-dropout architecture swap, not a tuning choice to flatter the results.
L1_VALUE = 0.001


def build_mt_mcdropout_model(n_features, l1_value=L1_VALUE, learning_rate=0.001, dropout_rate=0.2, seed=0):
    """MT's shared trunk (4 hard-sharing layers, 32 units) and factor-specific heads (2 layers,
    8 units, per factor) are unchanged from build_mt_model in Initial MT.ipynb. The only
    architectural change: batch norm -> dropout in the shared trunk, so dropout can be forced
    on at prediction time (training=True) without also perturbing batch norm's statistics."""
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inputs = Input(shape=(n_features,), name='predictors')
    x = inputs
    for i in range(4):
        x = layers.Dense(32, kernel_regularizer=regularizers.l1(l1_value), name=f'shared_dense_{i+1}')(x)
        x = layers.ReLU(name=f'shared_relu_{i+1}')(x)
        x = layers.Dropout(dropout_rate, name=f'shared_dropout_{i+1}')(x)
    shared_latent = x

    outputs = []
    for factor in FACTOR_NAMES:
        f = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'{factor}_dense1')(shared_latent)
        f = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'{factor}_dense2')(f)
        f_out = layers.Dense(1, activation='sigmoid', name=f'{factor}_output')(f)
        outputs.append(f_out)

    model = Model(inputs=inputs, outputs=outputs, name='MT_MCDropout')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={f'{factor}_output': 'binary_crossentropy' for factor in FACTOR_NAMES},
    )
    return model


def mc_dropout_predict(model, X, n_samples=N_MC_SAMPLES):
    """Run n_samples stochastic forward passes with dropout forced on. Returns an array of
    shape (n_samples, n_rows, n_factors); take .mean(axis=0) for the point estimate and
    .std(axis=0) for the uncertainty."""
    X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
    samples = [
        np.concatenate([np.asarray(p) for p in model(X_tensor, training=True)], axis=1)
        for _ in range(n_samples)
    ]
    return np.stack(samples, axis=0)


# --- sanity check with fake data, same pattern as Initial MT.ipynb's build_mt_model check ---
n_features = len(feature_cols)
model = build_mt_mcdropout_model(n_features)
X_fake = np.random.randn(20, n_features).astype('float32')
mc_fake = mc_dropout_predict(model, X_fake, n_samples=10)
print("MC sample array shape (n_samples, n_rows, n_factors):", mc_fake.shape)
print("Model builds and MC-samples without error.")

MC sample array shape (n_samples, n_rows, n_factors): (10, 20, 5)
Model builds and MC-samples without error.


In [4]:
# Train: identical walk-forward loop to Initial MT.ipynb's train cell (expanding window,
# 2-year validation window before the test year, single seed, fixed hyperparameters — same
# tractability simplification vs. the paper's grid search + 10-seed ensemble noted there).
#
# The only addition: after training, run MC Dropout on the validation set to calibrate this
# fold's uncertainty cutoff (validation only, never test — avoids leaking test-period info into
# the flagging rule), then run it on the test set to get this fold's point estimates + uncertainty.
# Also record this fold's validation-set std min/max per factor — used later to normalize the
# confidence-scaled strategy's weights per fold instead of over the pooled OOS series (leakage fix)
# — and this fold's validation-set *conviction* cutoff (median |prob - 0.5|, pooled across factors
# and months, same pooling convention as the uncertainty cutoff), used by the conviction-gated
# abstention strategy below.

mean_records, std_records, flag_records = [], [], []
bound_min_records, bound_max_records = [], []
conviction_cutoff_records = []

for test_year in est.OOS_TEST_YEARS:
    train, val, test, X_train, X_val, X_test = est.prepare_fold(data, feature_cols, test_year)

    y_train = {f'{f}_output': train[f'{f}_label'].values.astype('float32') for f in FACTOR_NAMES}
    y_val = {f'{f}_output': val[f'{f}_label'].values.astype('float32') for f in FACTOR_NAMES}

    model = build_mt_mcdropout_model(n_features=X_train.shape[1])
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(
        X_train.values.astype('float32'), y_train,
        validation_data=(X_val.values.astype('float32'), y_val),
        epochs=200, batch_size=4, callbacks=[es], verbose=0,
    )

    val_mc = mc_dropout_predict(model, X_val.values)
    val_mean, val_std = val_mc.mean(axis=0), val_mc.std(axis=0)  # (n_val, n_factors) each
    cutoff = np.quantile(val_std.flatten(), UNCERTAINTY_QUANTILE)
    val_std_min, val_std_max = val_std.min(axis=0), val_std.max(axis=0)  # per factor, this fold's val-only bounds
    val_conviction = np.abs(val_mean - 0.5)
    conviction_cutoff = np.median(val_conviction.flatten())  # pooled across factors+months, like `cutoff` above

    test_mc = mc_dropout_predict(model, X_test.values)
    test_mean, test_std = test_mc.mean(axis=0), test_mc.std(axis=0)
    test_flag = test_std > cutoff

    cols = [f'{f}_prob' for f in FACTOR_NAMES], [f'{f}_std' for f in FACTOR_NAMES], [f'{f}_uncertain' for f in FACTOR_NAMES]
    mean_records.append(pd.DataFrame(test_mean, index=test.index, columns=cols[0]))
    std_records.append(pd.DataFrame(test_std, index=test.index, columns=cols[1]))
    flag_records.append(pd.DataFrame(test_flag, index=test.index, columns=cols[2]))
    bound_min_records.append(pd.DataFrame(np.tile(val_std_min, (len(test), 1)), index=test.index, columns=cols[1]))
    bound_max_records.append(pd.DataFrame(np.tile(val_std_max, (len(test), 1)), index=test.index, columns=cols[1]))
    conviction_cutoff_records.append(pd.DataFrame(
        np.full((len(test), len(FACTOR_NAMES)), conviction_cutoff), index=test.index, columns=cols[1]
    ))

    print(f'{test_year}: cutoff={cutoff:.4f}  mean test uncertainty={test_std.mean():.4f}  '
          f'flagged={test_flag.mean():.0%}  conviction_cutoff={conviction_cutoff:.4f}')

oos_mean = pd.concat(mean_records).sort_index()
oos_std = pd.concat(std_records).sort_index()
oos_flag = pd.concat(flag_records).sort_index()
oos_std_val_min = pd.concat(bound_min_records).sort_index()  # this fold's val-set std min, per factor, broadcast to test rows
oos_std_val_max = pd.concat(bound_max_records).sort_index()  # same, max
oos_conviction_cutoff = pd.concat(conviction_cutoff_records).sort_index()  # this fold's val-set conviction cutoff, broadcast to test rows

oos_mean.to_csv('../results/mcdropout_oos_predictions.csv')
oos_std.to_csv('../results/mcdropout_oos_uncertainty.csv')
oos_flag.to_csv('../results/mcdropout_oos_flag.csv')
oos_conviction_cutoff.to_csv('../results/mcdropout_oos_conviction_cutoff.csv')
print("\nOOS predictions:", oos_mean.shape)

1990: cutoff=0.0564  mean test uncertainty=0.0562  flagged=40%  conviction_cutoff=0.0587


1991: cutoff=0.0655  mean test uncertainty=0.0563  flagged=30%  conviction_cutoff=0.0640


1992: cutoff=0.0855  mean test uncertainty=0.0459  flagged=12%  conviction_cutoff=0.0508


1993: cutoff=0.1184  mean test uncertainty=0.0691  flagged=12%  conviction_cutoff=0.0627


1994: cutoff=0.0670  mean test uncertainty=0.0434  flagged=7%  conviction_cutoff=0.0577


...

2017: cutoff=0.0728  mean test uncertainty=0.0444  flagged=18%  conviction_cutoff=0.0424


2018: cutoff=0.1069  mean test uncertainty=0.0681  flagged=30%  conviction_cutoff=0.0950


2019: cutoff=0.1175  mean test uncertainty=0.0866  flagged=27%  conviction_cutoff=0.0842


2020: cutoff=0.0972  mean test uncertainty=0.1360  flagged=60%  conviction_cutoff=0.0950


2021: cutoff=0.0843  mean test uncertainty=0.0752  flagged=42%  conviction_cutoff=0.0467

OOS predictions: (383, 5)


In [5]:
# Build three trading strategies from the same MC-Dropout predictions, to see whether the
# uncertainty signal is worth acting on:
#   baseline  - MT's usual rule: long if predicted prob > 0.5, flat otherwise (paper Eq. 3)
#   scaled    - baseline, but position size shrinks with uncertainty (gentler: smaller bet, not zero)
#   abstain   - baseline, but flagged (top 20% most uncertain) months are skipped entirely

response = loading.response_factors.rename(columns={'Mom': 'MOM'})
# a signal formed at t predicts sign(r_{t+1}) (see build_labels_and_panel / est.strategy_returns'
# docstring), so it must be graded against t+1's realized return, not t's -- shift(-1) here keeps
# this notebook's inline r consistent with est.strategy_returns, which every other notebook uses.
r = response.shift(-1).loc[oos_mean.index, FACTOR_NAMES]
assert not r.isna().any().any(), (
    'r contains NaN -- oos_mean includes a signal date at or after the last date in '
    'loading.response_factors, which has no following-month return to grade against.'
)
signal = pd.DataFrame({f: (oos_mean[f'{f}_prob'] > 0.5).astype(int) for f in FACTOR_NAMES}, index=oos_mean.index)

baseline_strat = signal * r
baseline_strat['EW'] = baseline_strat[FACTOR_NAMES].mean(axis=1)

# per-fold, per-factor min-max normalize uncertainty to a [0, 1] confidence weight (1 = certain,
# 0 = most uncertain), using each row's own fold's *validation-set* std min/max (oos_std_val_min/
# oos_std_val_max, computed in the training loop above) — never the test set or the pooled OOS
# series, which would leak knowledge of the full-sample uncertainty distribution into early test
# years. A fold whose validation set has zero spread for a factor can't be discriminated between
# certain/uncertain months, so it defaults to full confidence for that fold (same guard as before,
# now applied per fold instead of globally).
std_vals = oos_std.rename(columns=lambda c: c.replace('_std', ''))
bounds_min = oos_std_val_min.rename(columns=lambda c: c.replace('_std', ''))
bounds_max = oos_std_val_max.rename(columns=lambda c: c.replace('_std', ''))
value_range = (bounds_max - bounds_min).replace(0, 1)
norm_std = (std_vals - bounds_min) / value_range
confidence_weight = (1 - norm_std).clip(0, 1)

scaled_strat = signal * confidence_weight * r
scaled_strat['EW'] = scaled_strat[FACTOR_NAMES].mean(axis=1)

flag_vals = oos_flag.rename(columns=lambda c: c.replace('_uncertain', '')).astype(bool)
abstain_strat = signal * (~flag_vals).astype(int) * r
abstain_strat['EW'] = abstain_strat[FACTOR_NAMES].mean(axis=1)

print("mean confidence weight on flagged vs. unflagged factor-months:")
print(f"  flagged:   {confidence_weight.values[flag_vals.values].mean():.2f}")
print(f"  unflagged: {confidence_weight.values[~flag_vals.values].mean():.2f}")

mean confidence weight on flagged vs. unflagged factor-months:
  flagged:   0.26
  unflagged: 0.69


In [6]:
# --- Vol-regime abstain: an independent volatility-regime overlay, NOT another MC-Dropout
# variant. A prior diagnostic found the worst baseline losses (Feb 2000 SMB, Dec 2008 HML, the
# 2009 momentum-crash months, May 2021 HML) happen in HIGH-conviction, LOW-uncertainty months --
# MC Dropout's 50 stochastic passes all learned the same about-to-break pattern together, so
# nothing about the model's own self-assessment flags them. Conviction-gating was rejected for
# the same reason (it would keep exactly these high-conviction positions at full size).
#
# This strategy doesn't use MC Dropout uncertainty or conviction anywhere -- it abstains purely
# on trailing realized return volatility, a model-free, look-ahead-free feature of the return
# series itself, to test whether an independent regime signal catches what the model's own
# self-assessment missed.
#
# Trailing vol: rolling 3-month std of each factor's own monthly return, using only returns
# already realized by signal date t (r_t, r_{t-1}, r_{t-2} -- no look-ahead; NOT the shift(-1)
# used for r above, since here we want the vol already known *at* t, not the t+1 target).
# Cutoff calibrated per fold from that fold's *validation*-period trailing vol only (same
# UNCERTAINTY_QUANTILE discipline as everywhere else in this notebook) -- never from test.
# Threshold/window were fixed in advance (3-month, 80th percentile) and not tuned against
# these results.

raw_response = loading.response_factors.rename(columns={'Mom': 'MOM'})[FACTOR_NAMES]
trailing_vol = raw_response.rolling(3).std()

vol_flag_records = []
for test_year in est.OOS_TEST_YEARS:
    train, val, test, X_train, X_val, X_test = est.prepare_fold(data, feature_cols, test_year)
    val_vol = trailing_vol.loc[val.index, FACTOR_NAMES]
    vol_cutoff = np.quantile(val_vol.values.flatten(), UNCERTAINTY_QUANTILE)
    test_vol = trailing_vol.loc[test.index, FACTOR_NAMES]
    vol_flag_records.append(pd.DataFrame((test_vol > vol_cutoff).values, index=test.index, columns=FACTOR_NAMES))

oos_vol_flag = pd.concat(vol_flag_records).sort_index()
oos_vol_flag.to_csv('../results/mcdropout_oos_vol_flag.csv')

vol_regime_strat = signal * (~oos_vol_flag).astype(int) * r
vol_regime_strat['EW'] = vol_regime_strat[FACTOR_NAMES].mean(axis=1)

print(f"vol-regime abstain: flagged {oos_vol_flag.values.mean():.1%} of factor-months")

vol-regime abstain: flagged 22.5% of factor-months


In [7]:
# Benchmark: does acting on MC-Dropout uncertainty improve risk-adjusted performance relative
# to the unweighted baseline, and is the uncertainty signal actually informative (do flagged
# months really predict worse, i.e. lower accuracy)?

# same t -> t+1 alignment as r above, so the spanning regression pairs each strategy's return
# against the BUY benchmark's return in the same realized month, not one month earlier.
buy_ew = response.shift(-1).loc[oos_mean.index, FACTOR_NAMES].mean(axis=1)

# accuracy restricted to factor-months where each strategy actually held a nonzero position --
# months a strategy sat out (or, for confidence-scaled, sized down to near-zero) aren't a
# directional call, so folding them into accuracy would dilute the number with non-decisions.
# confidence-scaled weights are continuous rather than 0/1, so "held a position" needs an
# explicit small threshold rather than a literal nonzero check.
y_true = pd.DataFrame({f: data.loc[oos_mean.index, f'{f}_label'] for f in FACTOR_NAMES})
correct = (y_true == signal)

POSITION_WEIGHT_THRESHOLD = 0.05  # confidence-scaled weight below this counts as "no position"
baseline_position = signal == 1
scaled_position = (signal == 1) & (confidence_weight > POSITION_WEIGHT_THRESHOLD)
abstain_position = (signal == 1) & (~flag_vals)
vol_position = (signal == 1) & (~oos_vol_flag)

def accuracy_when_positioned(position_mask):
    return correct.values[position_mask.values].mean() * 100

perf = pd.DataFrame({
    'baseline (no uncertainty)': [
        est.annualized_sharpe(baseline_strat['EW']), *est.spanning_regression(baseline_strat['EW'], buy_ew),
        accuracy_when_positioned(baseline_position),
    ],
    'confidence-scaled': [
        est.annualized_sharpe(scaled_strat['EW']), *est.spanning_regression(scaled_strat['EW'], buy_ew),
        accuracy_when_positioned(scaled_position),
    ],
    'abstain on flagged': [
        est.annualized_sharpe(abstain_strat['EW']), *est.spanning_regression(abstain_strat['EW'], buy_ew),
        accuracy_when_positioned(abstain_position),
    ],
    'vol-regime abstain': [
        est.annualized_sharpe(vol_regime_strat['EW']), *est.spanning_regression(vol_regime_strat['EW'], buy_ew),
        accuracy_when_positioned(vol_position),
    ],
}, index=['Sharpe Ratio', 'alpha (annualized %)', 't(alpha)', 'beta', 'R2 (%)', 'Accuracy when positioned (%)'])
print("Multi-factor timing performance vs. multi-factor BUY:")
print(perf.round(3).to_string())
print(f"\nmulti-factor BUY Sharpe: {est.annualized_sharpe(buy_ew):.2f}")

# calibration check: if MC-Dropout uncertainty is meaningful, flagged (high-uncertainty)
# factor-months should have *lower* classification accuracy than unflagged ones.
acc_flagged = correct.values[flag_vals.values].mean()
acc_unflagged = correct.values[~flag_vals.values].mean()
print(f"\nAccuracy on flagged (uncertain) factor-months:   {acc_flagged:.1%}  (n={flag_vals.values.sum()})")
print(f"Accuracy on unflagged (confident) factor-months: {acc_unflagged:.1%}  (n={(~flag_vals.values).sum()})")

# vol-regime abstain: did it actually catch the specific months that broke the baseline?
print("\nvol-regime abstain -- hit/miss on the four worst diagnosed losses:")
named_months = [('2000-02-29', 'SMB'), ('2008-12-31', 'HML'), ('2009-04-30', 'MOM'), ('2021-05-31', 'HML')]
for dt, f in named_months:
    hit = bool(oos_vol_flag.loc[dt, f])
    print(f"  {dt} {f}: {'ABSTAINED (hit)' if hit else 'still exposed (miss)'}")

# does trailing-vol flag genuinely different months than MC-Dropout uncertainty, or just the
# same ones by another name?
vol_flagged_mask = oos_vol_flag.values
overlap_with_mc = flag_vals.values[vol_flagged_mask].mean()
print(f"\nOf all vol-regime-flagged factor-months (n={vol_flagged_mask.sum()}), "
      f"{overlap_with_mc:.1%} were also MC-Dropout-flagged as uncertain "
      f"({1 - overlap_with_mc:.1%} were months MC-Dropout was confident about).")

Multi-factor timing performance vs. multi-factor BUY:
                              baseline (no uncertainty)  confidence-scaled  abstain on flagged  vol-regime abstain
Sharpe Ratio                                      0.631              0.508               0.588               0.613
alpha (annualized %)                              0.338              0.078               0.753               0.868
t(alpha)                                          1.107              0.260               1.350               1.503
beta                                              0.866              0.474               0.592               0.435
R2 (%)                                           86.738             64.376              47.576              37.900
Accuracy when positioned (%)                     54.921             55.172              56.525              54.638

multi-factor BUY Sharpe: 0.60

Accuracy on flagged (uncertain) factor-months:   51.3%  (n=476)
Accuracy on unflagged (confident) factor-mont